# Praktikum Bab VIII — Agentic AI## Membangun Agentic RAG untuk Tanya-Jawab Dokumen Berbahasa Indonesia**TEMPLATE PENGERJAAN**Lengkapi setiap bagian yang ditandai `TODO`. Jalankan notebook dari atas ke bawah.Semua fungsi yang sudah disediakan boleh diubah selama hasil akhirnya setara.Nama / NIM  : ...Kelompok    : ...Tanggal     : ...| Langkah | Isi | Status ||---|---|---|| 1 | Verifikasi lingkungan | [ ] || 2 | Korpus dan dataset uji | [ ] || 3 | Chunking dan indexing | [ ] || 4 | Retrieval dan pengukuran | [ ] || 5 | Generator RAG | [ ] || 6 | Loop ReAct | [ ] || 7 | Agent verifikator | [ ] || 8 | Evaluasi | [ ] || 9 | Keamanan | [ ] |

---## Langkah 1 — Verifikasi LingkunganJalankan tiga sel berikut sebelum melanjutkan. Jangan lanjut ke Langkah 2 sebelumketiganya lolos: pustaka terpasang, model embedding bisa dimuat, dan LLM bisa dipanggil.

In [ ]:
import os, re, json, time, math, randomfrom pathlib import Pathfrom collections import defaultdictimport numpy as npSEED = 42random.seed(SEED)np.random.seed(SEED)BASE = Path(".")DIR_KORPUS = BASE / "data" / "korpus"DIR_KORPUS_DISUSUPI = BASE / "data" / "korpus_disusupi"DIR_HASIL = BASE / "hasil"for d in (DIR_KORPUS, DIR_KORPUS_DISUSUPI, DIR_HASIL):    d.mkdir(parents=True, exist_ok=True)def simpan_hasil(nama, obj):    # Simpan keluaran tiap langkah supaya bisa dilampirkan ke laporan    p = DIR_HASIL / f"{nama}.json"    p.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")    print("tersimpan:", p)try:    from dotenv import load_dotenv    load_dotenv()                      # baca berkas .env bila adaexcept ImportError:    passimport sysprint("Python:", sys.version.split()[0])for mod in ["sentence_transformers", "chromadb", "rank_bm25"]:    try:        __import__(mod)        print(f"  {mod}: OK")    except ImportError:        print(f"  {mod}: BELUM TERPASANG -> pip install -r requirements.txt")

In [ ]:
# Lapisan tipis di atas LLM supaya bisa berpindah antara model lokal (Ollama) dan# API Gemini tanpa mengubah kode di bagian lain.## Atur lewat berkas .env:#   LLM_BACKEND=ollama   LLM_MODEL=qwen2.5:7b-instruct#   LLM_BACKEND=gemini   LLM_MODEL=gemini-2.5-flash   GEMINI_API_KEY=AIza..._MODEL_BAWAAN = {"ollama": "qwen2.5:7b-instruct", "gemini": "gemini-2.5-flash"}LLM_BACKEND = os.getenv("LLM_BACKEND", "ollama").lower()LLM_MODEL   = os.getenv("LLM_MODEL") or _MODEL_BAWAAN.get(LLM_BACKEND, "qwen2.5:7b-instruct")# Kuota gratis Gemini dibatasi per menit DAN per hari. Sesuaikan angka ini dengan# batas yang tertera di https://aistudio.google.com/rate-limitRPM_MAKS = int(os.getenv("RPM_MAKS", "10"))TOTAL_TOKEN = {"masuk": 0, "keluar": 0, "panggilan": 0}_KLIEN_GEMINI = None_JEDA_MIN = 60.0 / max(RPM_MAKS, 1)_waktu_terakhir = [0.0]def _tahan_laju():    # Jeda sederhana supaya tidak menabrak batas permintaan per menit    selisih = time.time() - _waktu_terakhir[0]    if selisih < _JEDA_MIN:        time.sleep(_JEDA_MIN - selisih)    _waktu_terakhir[0] = time.time()def _chat_ollama(messages, temperature, max_tokens):    import ollama    r = ollama.chat(        model=LLM_MODEL,        messages=messages,        options={"temperature": temperature, "num_predict": max_tokens, "seed": SEED},    )    TOTAL_TOKEN["masuk"]  += r.get("prompt_eval_count", 0) or 0    TOTAL_TOKEN["keluar"] += r.get("eval_count", 0) or 0    return r["message"]["content"]def _chat_gemini(messages, temperature, max_tokens):    from google import genai    from google.genai import types    global _KLIEN_GEMINI    if _KLIEN_GEMINI is None:        kunci = os.getenv("GEMINI_API_KEY")        if not kunci:            raise RuntimeError(                "GEMINI_API_KEY belum diset. Buat key di https://aistudio.google.com "                "lalu simpan di berkas .env."            )        _KLIEN_GEMINI = genai.Client(api_key=kunci)    # Gemini memisahkan system instruction dari isi percakapan    sistem = "\n".join(m["content"] for m in messages if m["role"] == "system") or None    isi = [        types.Content(            role=("model" if m["role"] == "assistant" else "user"),            parts=[types.Part(text=m["content"])],        )        for m in messages if m["role"] != "system"    ]    opsi = dict(system_instruction=sistem, temperature=temperature,                max_output_tokens=max_tokens)    # Matikan mode berpikir bila tersedia. Tanpa ini, token keluaran bisa habis    # dipakai proses berpikir sehingga teks jawabannya kosong.    try:        cfg = types.GenerateContentConfig(            seed=SEED, thinking_config=types.ThinkingConfig(thinking_budget=0), **opsi)    except (TypeError, AttributeError, ValueError):        try:            cfg = types.GenerateContentConfig(seed=SEED, **opsi)        except TypeError:            cfg = types.GenerateContentConfig(**opsi)    r = _KLIEN_GEMINI.models.generate_content(model=LLM_MODEL, contents=isi, config=cfg)    u = getattr(r, "usage_metadata", None)    if u:        TOTAL_TOKEN["masuk"]  += getattr(u, "prompt_token_count", 0) or 0        TOTAL_TOKEN["keluar"] += getattr(u, "candidates_token_count", 0) or 0    return (r.text or "").strip()def chat(messages, temperature=0.0, max_tokens=1024, maks_coba=4):    for percobaan in range(maks_coba):        try:            if LLM_BACKEND == "gemini":                _tahan_laju()            TOTAL_TOKEN["panggilan"] += 1            if LLM_BACKEND == "ollama":                return _chat_ollama(messages, temperature, max_tokens)            if LLM_BACKEND == "gemini":                return _chat_gemini(messages, temperature, max_tokens)            raise ValueError(f"LLM_BACKEND tidak dikenal: {LLM_BACKEND}")        except Exception as e:            pesan = str(e).upper()            kena_batas = "429" in pesan or "RESOURCE_EXHAUSTED" in pesan or "RATE" in pesan            if not kena_batas or percobaan == maks_coba - 1:                raise            jeda = 5 * (2 ** percobaan)            print(f"[kuota] kena batas laju, menunggu {jeda} detik lalu mencoba lagi...")            time.sleep(jeda)def tanya(prompt, sistem=None, **kw):    msgs = []    if sistem:        msgs.append({"role": "system", "content": sistem})    msgs.append({"role": "user", "content": prompt})    return chat(msgs, **kw)print(f"backend: {LLM_BACKEND} | model: {LLM_MODEL}"      + (f" | jeda antar-panggilan: {_JEDA_MIN:.1f} detik" if LLM_BACKEND == "gemini" else ""))print(tanya("Sebutkan tiga kota di Jawa Barat. Jawab singkat saja."))

In [ ]:
from sentence_transformers import SentenceTransformerNAMA_MODEL_EMBED = "intfloat/multilingual-e5-small"model_embed = SentenceTransformer(NAMA_MODEL_EMBED)# Model keluarga E5 MEWAJIBKAN awalan. Tanpa ini kualitas pencarian turun jauh.def embed_dokumen(teks_list):    teks = [f"passage: {t}" for t in teks_list]    return model_embed.encode(teks, normalize_embeddings=True, show_progress_bar=False)def embed_query(teks):    return model_embed.encode([f"query: {teks}"], normalize_embeddings=True)[0]v = embed_query("uji coba")print("dimensi embedding:", v.shape, "| norma:", round(float(np.linalg.norm(v)), 4))

---## Langkah 2 — Korpus dan Dataset UjiKorpus contoh di bawah adalah peraturan akademik fiktif, disediakan supaya notebookbisa langsung dijalankan. Ganti dengan korpus asli dengan meletakkan berkas `.txt`berencoding UTF-8 di `data/korpus/`.Dataset uji wajib memuat empat tipe pertanyaan: `istilah_spesifik`, `parafrase`,`multi_hop`, dan `tidak_terjawab`.

In [ ]:
# Korpus contoh: peraturan akademik fiktif, supaya notebook bisa langsung dijalankan.# Ganti dengan korpus asli dengan meletakkan berkas .txt di data/korpus/KORPUS_CONTOH = {}KORPUS_CONTOH["peraturan_tugas_akhir.txt"] = ("Peraturan Tugas Akhir Program Sarjana\n\n""Mahasiswa dapat mengambil mata kuliah Tugas Akhir apabila telah menempuh ""sekurang-kurangnya 110 SKS dengan Indeks Prestasi Kumulatif minimal 2,00.\n\n""Tugas Akhir terdiri atas dua tahap, yaitu Tugas Akhir I dengan bobot 2 SKS dan ""Tugas Akhir II dengan bobot 4 SKS. Tugas Akhir I berisi penyusunan proposal dan ""studi literatur, sedangkan Tugas Akhir II berisi pelaksanaan penelitian dan ""penulisan laporan.\n\n""Masa pengerjaan Tugas Akhir paling lama adalah dua semester berturut-turut. ""Perpanjangan hanya dapat diberikan satu kali atas persetujuan dosen pembimbing ""dan Ketua Program Studi.\n\n""Setiap mahasiswa dibimbing oleh sekurang-kurangnya satu dosen pembimbing. ""Penggantian dosen pembimbing dapat diajukan paling lambat pada akhir minggu ""keempat semester berjalan.")KORPUS_CONTOH["peraturan_sidang.txt"] = ("Peraturan Sidang Tugas Akhir\n\n""Sidang Tugas Akhir dapat dilaksanakan apabila mahasiswa telah menyelesaikan ""seluruh mata kuliah wajib dan memperoleh persetujuan tertulis dari dosen ""pembimbing.\n\n""Majelis sidang terdiri atas satu orang dosen pembimbing dan dua orang dosen ""penguji. Sidang dinyatakan sah apabila dihadiri sekurang-kurangnya tiga orang ""anggota majelis.\n\n""Nilai kelulusan sidang minimal adalah C. Mahasiswa yang memperoleh nilai di ""bawah C wajib mengulang sidang paling cepat empat minggu setelah sidang ""pertama.\n\n""Berkas sidang diserahkan ke Tata Usaha paling lambat tujuh hari kerja sebelum ""tanggal pelaksanaan sidang.")KORPUS_CONTOH["peraturan_kelulusan.txt"] = ("Syarat Kelulusan Program Sarjana\n\n""Mahasiswa dinyatakan lulus apabila telah menempuh sekurang-kurangnya 144 SKS ""dengan Indeks Prestasi Kumulatif minimal 2,00 dan tidak memiliki nilai E.\n\n""Predikat kelulusan ditetapkan sebagai berikut. Predikat Memuaskan untuk IPK ""2,00 sampai 2,75. Predikat Sangat Memuaskan untuk IPK 2,76 sampai 3,50. ""Predikat Dengan Pujian untuk IPK di atas 3,50 dengan masa studi tidak lebih ""dari sepuluh semester.\n\n""Mahasiswa wajib menyerahkan bukti bebas pinjaman perpustakaan dan bebas ""tanggungan laboratorium sebelum yudisium.")KORPUS_CONTOH["peraturan_cuti.txt"] = ("Peraturan Cuti Akademik\n\n""Cuti akademik dapat diajukan mahasiswa yang telah menempuh sekurang-kurangnya ""dua semester. Pengajuan dilakukan paling lambat dua minggu sebelum masa ""perwalian dimulai.\n\n""Cuti akademik diberikan paling lama dua semester, baik berturut-turut maupun ""tidak, selama masa studi.\n\n""Masa cuti akademik tidak diperhitungkan dalam masa studi. Mahasiswa yang ""berhenti kuliah tanpa mengajukan cuti tetap diperhitungkan masa studinya dan ""dikenakan biaya penuh.")KORPUS_CONTOH["peraturan_perwalian.txt"] = ("Peraturan Perwalian dan Rencana Studi\n\n""Perwalian dilaksanakan pada awal setiap semester. Mahasiswa wajib berkonsultasi ""dengan dosen wali sebelum menyusun Rencana Studi.\n\n""Beban studi maksimum ditentukan oleh Indeks Prestasi semester sebelumnya. ""IP di bawah 2,00 memperoleh beban maksimum 15 SKS. IP 2,00 sampai 2,99 ""memperoleh 18 SKS. IP 3,00 ke atas memperoleh 24 SKS.\n\n""Perubahan Rencana Studi hanya dapat dilakukan pada dua minggu pertama ""perkuliahan melalui persetujuan dosen wali.")KORPUS_CONTOH["panduan_kerja_praktik.txt"] = ("Panduan Kerja Praktik\n\n""Kerja Praktik berbobot 2 SKS dan dapat diambil setelah mahasiswa menempuh ""sekurang-kurangnya 80 SKS.\n\n""Durasi pelaksanaan Kerja Praktik paling singkat adalah 30 hari kerja di ""instansi mitra. Laporan Kerja Praktik diserahkan paling lambat satu bulan ""setelah pelaksanaan berakhir.\n\n""Penilaian Kerja Praktik terdiri atas penilaian pembimbing lapangan dengan ""bobot 40 persen dan penilaian dosen pembimbing dengan bobot 60 persen.")def tulis_korpus_contoh(folder):    folder.mkdir(parents=True, exist_ok=True)    for nama, isi in KORPUS_CONTOH.items():        (folder / nama).write_text(isi, encoding="utf-8")def muat_korpus(folder):    berkas = sorted(folder.glob("*.txt"))    return {p.name: p.read_text(encoding="utf-8") for p in berkas}if not list(DIR_KORPUS.glob("*.txt")):    tulis_korpus_contoh(DIR_KORPUS)    print("korpus contoh ditulis ke", DIR_KORPUS)korpus = muat_korpus(DIR_KORPUS)print(f"{len(korpus)} dokumen dimuat, total {sum(len(t.split()) for t in korpus.values())} kata")

In [ ]:
# Dataset uji. Untuk praktikum sebenarnya, berkas ini disiapkan asisten supaya# angka antar-kelompok bisa dibandingkan. Minimal 30 pertanyaan, empat tipe terwakili.DATASET_CONTOH = [ {"id":"Q01","pertanyaan":"Berapa SKS minimum untuk mengambil Tugas Akhir?",  "jawaban_acuan":"110 SKS","dokumen_kunci":["peraturan_tugas_akhir.txt"],"tipe":"istilah_spesifik"}, {"id":"Q02","pertanyaan":"Berapa bobot SKS Tugas Akhir II?",  "jawaban_acuan":"4 SKS","dokumen_kunci":["peraturan_tugas_akhir.txt"],"tipe":"istilah_spesifik"}, {"id":"Q03","pertanyaan":"Berapa orang yang menguji saat sidang?",  "jawaban_acuan":"Dua orang dosen penguji","dokumen_kunci":["peraturan_sidang.txt"],"tipe":"parafrase"}, {"id":"Q04","pertanyaan":"Kalau nilai sidang saya jelek, kapan boleh coba lagi?",  "jawaban_acuan":"Paling cepat empat minggu setelah sidang pertama","dokumen_kunci":["peraturan_sidang.txt"],"tipe":"parafrase"}, {"id":"Q05","pertanyaan":"Berapa IPK minimal untuk lulus?",  "jawaban_acuan":"2,00","dokumen_kunci":["peraturan_kelulusan.txt"],"tipe":"istilah_spesifik"}, {"id":"Q06","pertanyaan":"Apa syarat mendapat predikat Dengan Pujian?",  "jawaban_acuan":"IPK di atas 3,50 dan masa studi tidak lebih dari sepuluh semester",  "dokumen_kunci":["peraturan_kelulusan.txt"],"tipe":"istilah_spesifik"}, {"id":"Q07","pertanyaan":"Saya ingin berhenti kuliah sementara, apa yang harus diurus?",  "jawaban_acuan":"Mengajukan cuti akademik paling lambat dua minggu sebelum perwalian",  "dokumen_kunci":["peraturan_cuti.txt"],"tipe":"parafrase"}, {"id":"Q08","pertanyaan":"Berapa lama maksimal cuti akademik selama masa studi?",  "jawaban_acuan":"Dua semester","dokumen_kunci":["peraturan_cuti.txt"],"tipe":"istilah_spesifik"}, {"id":"Q09","pertanyaan":"Kalau IP semester lalu 3,2 boleh ambil berapa SKS?",  "jawaban_acuan":"24 SKS","dokumen_kunci":["peraturan_perwalian.txt"],"tipe":"parafrase"}, {"id":"Q10","pertanyaan":"Berapa lama minimal pelaksanaan Kerja Praktik?",  "jawaban_acuan":"30 hari kerja","dokumen_kunci":["panduan_kerja_praktik.txt"],"tipe":"istilah_spesifik"}, {"id":"Q11","pertanyaan":"Berapa total SKS Tugas Akhir dan Kerja Praktik kalau digabung?",  "jawaban_acuan":"8 SKS (2+4 untuk TA dan 2 untuk KP)",  "dokumen_kunci":["peraturan_tugas_akhir.txt","panduan_kerja_praktik.txt"],"tipe":"multi_hop"}, {"id":"Q12","pertanyaan":"Kalau sudah 110 SKS, apakah sudah boleh ambil Kerja Praktik sekaligus Tugas Akhir?",  "jawaban_acuan":"Ya, karena syarat KP 80 SKS dan syarat TA 110 SKS",  "dokumen_kunci":["peraturan_tugas_akhir.txt","panduan_kerja_praktik.txt"],"tipe":"multi_hop"}, {"id":"Q13","pertanyaan":"Berapa selisih SKS kelulusan dengan syarat minimum ambil Tugas Akhir?",  "jawaban_acuan":"34 SKS (144 dikurangi 110)",  "dokumen_kunci":["peraturan_kelulusan.txt","peraturan_tugas_akhir.txt"],"tipe":"multi_hop"}, {"id":"Q14","pertanyaan":"Berapa biaya UKT untuk mahasiswa angkatan tahun ini?",  "jawaban_acuan":"TIDAK ADA DI KORPUS","dokumen_kunci":[],"tipe":"tidak_terjawab"}, {"id":"Q15","pertanyaan":"Siapa nama Rektor saat ini?",  "jawaban_acuan":"TIDAK ADA DI KORPUS","dokumen_kunci":[],"tipe":"tidak_terjawab"},]BERKAS_UJI = BASE / "data" / "dataset_uji.json"if not BERKAS_UJI.exists():    BERKAS_UJI.write_text(json.dumps(DATASET_CONTOH, ensure_ascii=False, indent=2), encoding="utf-8")dataset_uji = json.loads(BERKAS_UJI.read_text(encoding="utf-8"))print(f"{len(dataset_uji)} pertanyaan uji")for t in sorted({d["tipe"] for d in dataset_uji}):    print(f"  {t}: {sum(1 for d in dataset_uji if d['tipe']==t)}")

---## Langkah 3 — Chunking dan IndexingDua metode yang diimplementasikan:1. **Fixed-size** — potong tiap N karakter dengan overlap tetap2. **Recursive** — potong mengikuti pemisah alami bertingkat: paragraf, kalimat, kataCatatan penting soal model E5: teks harus diberi awalan sebelum di-embed, yaitu`query: ` untuk pertanyaan dan `passage: ` untuk dokumen. Tanpa awalan itu kualitaspencarian turun cukup jauh.

In [ ]:
def fixed_chunk(teks, ukuran=500, overlap=50):    # TODO 3.1 — potong teks tiap `ukuran` karakter dengan `overlap` karakter    # tumpang tindih antar-potongan. Kembalikan list of string, buang yang kosong.    raise NotImplementedErrordef recursive_chunk(teks, ukuran=500, overlap=50, pemisah=None):    # TODO 3.2 — potong mengikuti struktur alami teks secara bertingkat.    # Urutan pemisah yang disarankan: ["\n\n", "\n", ". ", " "].    # Gabungkan bagian selama total panjangnya masih <= ukuran; kalau satu bagian    # sendiri sudah lebih panjang dari ukuran, pecah lagi dengan pemisah berikutnya.    raise NotImplementedErrordef bangun_chunk(korpus, metode="recursive", ukuran=500, overlap=50):    # Sudah disediakan. Tiap chunk wajib punya id unik, teks, dokumen asal,    # nomor urut, dan metode. Metadata ini dipakai untuk Recall@k dan sitasi.    fn = fixed_chunk if metode == "fixed" else recursive_chunk    keluaran = []    for nama_dok, teks in korpus.items():        for i, c in enumerate(fn(teks, ukuran, overlap)):            keluaran.append({                "id": f"{nama_dok}::{metode}::{i}",                "teks": c, "dokumen": nama_dok, "urut": i, "metode": metode,            })    return keluaran

In [ ]:
import pandas as pd# TODO 3.3 — bandingkan kedua metode. Untuk tiap metode catat:#   jumlah chunk, rata-rata panjang, terpendek, terpanjang, dan berapa chunk#   yang berakhir di tengah kalimat (tidak diakhiri tanda baca akhir).# Tampilkan sebagai DataFrame, lalu simpan dengan simpan_hasil("l3_perbandingan_chunking", ...)baris = []kandidat = {}for m in ["fixed", "recursive"]:    ...# TODO 3.4 — tentukan metode mana yang dipakai untuk langkah berikutnya,# dan tuliskan alasannya di sel markdown di bawah.chunks = None

**Jawaban TODO 3.4** — metode yang dipilih dan alasannya:_(tulis di sini)_

In [ ]:
import chromadbklien = chromadb.Client()try:    klien.delete_collection("korpus")except Exception:    passkoleksi = klien.create_collection("korpus", metadata={"hnsw:space": "cosine"})# TODO 3.5 — embed seluruh chunk dan masukkan ke koleksi.#   koleksi.add(ids=..., embeddings=..., documents=..., metadatas=...)#   Ingat: embeddings harus berupa list of list (pakai .tolist()).#   Metadata minimal berisi "dokumen" dan "urut".# TODO 3.6 — buktikan pengaruh awalan E5. Cari satu pertanyaan uji dua kali,# sekali dengan awalan "query: " dan sekali tanpa awalan, lalu bandingkan# dokumen mana yang terambil. Catat hasilnya di laporan.

---## Langkah 4 — Retrieval dan PengukurannyaEmpat konfigurasi dibangun bertahap dan diukur dengan dataset uji yang sama:1. Dense saja2. BM25 saja3. Hybrid dengan Reciprocal Rank Fusion4. Hybrid + re-rank cross-encoderMetrik: Recall@5, Recall@10, MRR, dan latensi.

In [ ]:
# --- 1. Dense ----------------------------------------------------------------def cari_dense(query, k=10):    # TODO 4.1 — embed query, panggil koleksi.query, kembalikan list of dict    # berisi id, teks, dokumen, dan skor (ingat Chroma mengembalikan distance,    # bukan similarity).    raise NotImplementedError# --- 2. BM25 -----------------------------------------------------------------from rank_bm25 import BM25Okapidef tokenisasi(teks):    # TODO 4.2 — tokenisasi sederhana: huruf kecil + pisah kata.    # Opsional: coba tambahkan stemming Sastrawi lalu bandingkan hasilnya.    raise NotImplementedError# TODO 4.3 — bangun indeks BM25 dari seluruh chunkbm25 = Nonedef cari_bm25(query, k=10):    # TODO 4.4 — hitung skor BM25, ambil k teratas, kembalikan bentuk yang sama    # dengan cari_dense supaya bisa dipertukarkan.    raise NotImplementedError# --- 3. Hybrid dengan Reciprocal Rank Fusion ---------------------------------def rrf(daftar_peringkat, k_rrf=60, k=10):    # TODO 4.5 — gabungkan beberapa daftar hasil berdasarkan POSISI peringkat.    # Rumusnya: skor(d) = sum over daftar of 1 / (k_rrf + peringkat_d).    # Perhatikan mengapa yang dipakai peringkat, bukan skor mentah.    raise NotImplementedErrordef cari_hybrid(query, k=10, k_kandidat=30):    # TODO 4.6 — ambil kandidat dari dense dan BM25, lalu satukan dengan rrf    raise NotImplementedError# --- 4. Hybrid + re-rank -----------------------------------------------------from sentence_transformers import CrossEncoder# TODO 4.7 — muat cross-encoder, misalnya "BAAI/bge-reranker-base"reranker = Nonedef cari_rerank(query, k=5, k_kandidat=30):    # TODO 4.8 — ambil k_kandidat dari hybrid, beri skor ulang dengan cross-encoder    # pada pasangan (query, teks), lalu kembalikan k teratas.    raise NotImplementedError

In [ ]:
def recall_at_k(hasil, kunci, k):    # TODO 4.9 — proporsi dokumen kunci yang muncul di k hasil teratas.    # Kembalikan None bila kunci kosong (pertanyaan tipe tidak_terjawab).    raise NotImplementedErrordef reciprocal_rank(hasil, kunci):    # TODO 4.10 — 1 dibagi posisi hasil relevan pertama, atau 0 bila tidak ada    raise NotImplementedErrorKONFIGURASI = {    "dense":         lambda q: cari_dense(q, k=10),    "bm25":          lambda q: cari_bm25(q, k=10),    "hybrid_rrf":    lambda q: cari_hybrid(q, k=10),    "hybrid_rerank": lambda q: cari_rerank(q, k=10, k_kandidat=30),}def evaluasi_retriever(dataset):    # TODO 4.11 — untuk tiap konfigurasi hitung Recall@5, Recall@10, MRR, dan    # latensi rata-rata. Kembalikan dua DataFrame: ringkasan dan rincian per soal.    # Rincian per soal dibutuhkan untuk analisis kegagalan di sel berikutnya.    raise NotImplementedError# tabel_retriever, detail_retriever = evaluasi_retriever(dataset_uji)# display(tabel_retriever)# simpan_hasil("l4_evaluasi_retriever", tabel_retriever.to_dict("records"))

In [ ]:
# TODO 4.12 (TUGAS LANGKAH 4) — analisis kegagalan.# Temukan minimal satu pertanyaan yang:#   (a) gagal di dense tapi berhasil di BM25#   (b) gagal di BM25 tapi berhasil di dense# Untuk tiap kasus, tampilkan tiga potongan teratas dari kedua metode,# lalu jelaskan penyebabnya di sel markdown dengan merujuk Sub Bab XIV.

**Jawaban TUGAS LANGKAH 4** — analisis kegagalan dense vs BM25:_(tulis di sini)_

---## Langkah 5 — Generator RAGPrompt wajib memuat tiga hal: perintah menjawab hanya dari konteks, perintah mengakutidak tahu bila konteksnya tidak memuat jawaban, dan perintah memberi sitasi`[nama_dokumen]` pada tiap klaim.

In [ ]:
# TODO 5.1 — susun system prompt. Wajib memuat tiga hal:#   1. hanya menjawab dari dokumen yang diberikan#   2. menjawab "Informasi tidak ditemukan dalam dokumen." bila tidak ada jawabannya#   3. memberi sitasi [nama_dokumen] pada tiap klaimSISTEM_RAG = ""def susun_konteks(potongan):    # TODO 5.2 — rangkai potongan menjadi satu blok konteks.    # Bungkus tiap potongan dengan penanda yang jelas, misalnya    # <dokumen sumber="nama.txt">...</dokumen>. Penanda ini akan dipakai lagi    # sebagai lapis pertama guardrail di Langkah 9, jadi rancang sejak sekarang.    raise NotImplementedErrordef jawab_rag(pertanyaan, k=5, retriever=None, urutan="relevan_dulu"):    # TODO 5.3 — ambil potongan, susun prompt, panggil LLM.    # Kembalikan (jawaban, potongan) supaya potongannya bisa dinilai di Langkah 8.    # Sediakan juga mode urutan="relevan_di_tengah" untuk eksperimen    # Lost in the Middle.    raise NotImplementedError

In [ ]:
# TODO 5.4 — jalankan seluruh dataset uji.# Hitung khusus: dari pertanyaan bertipe "tidak_terjawab", berapa yang TETAP# dijawab model dengan percaya diri? Angka itu ukuran halusinasi paling kasar.# Simpan hasilnya dengan simpan_hasil("l5_jawaban_rag", ...)# TODO 5.5 (opsional) — bandingkan urutan="relevan_dulu" dengan# urutan="relevan_di_tengah" dan catat selisihnya.

---## Langkah 6 — Dari RAG ke Agent (loop ReAct)Sampai Langkah 5 alurnya masih satu arah. Di sini model yang memutuskan kapan mencari.**Loop ReAct ditulis sendiri, tanpa framework.** Larangan ini disengaja: begituframework dipakai, yang terjadi di dalam satu putaran agent tidak pernah terlihat.Yang diimplementasikan: definisi tool, prompt ReAct, parser, loop eksekusi, batasputaran, dan penanganan kesalahan.

In [ ]:
# --- Definisi tool -----------------------------------------------------------def tool_cari_dokumen(query):    # TODO 6.1 — bungkus retriever terbaik dari Langkah 4 menjadi tool.    # Kembalikan STRING, karena hasilnya akan ditempel sebagai Observation.    raise NotImplementedErrordef tool_kalkulator(ekspresi):    # TODO 6.2 — hitung ekspresi aritmetika sederhana.    # PENTING: batasi masukan yang diizinkan. Jangan pernah meneruskan string    # apa pun ke eval tanpa penyaringan. Ini contoh kecil dari prinsip    # pembatasan akses tool di Sub Bab XVI.    raise NotImplementedError# TODO 6.3 — daftarkan tool beserta deskripsi dan skema parameternya.# Deskripsi inilah yang dibaca model untuk memilih tool, jadi tulis dengan jelas.TOOLS = {}def daftar_tool_teks():    raise NotImplementedError# --- Prompt ReAct ------------------------------------------------------------# TODO 6.4 — susun prompt dengan format bergantian:#   Thought / Action / Action Input / Observation, diakhiri Final Answer.# Sertakan juga aturan: jangan mengarang, satu Action per giliran, dan apa yang# harus ditulis bila jawabannya tidak ada di dokumen.PROMPT_REACT = ""# --- Parser ------------------------------------------------------------------def urai_langkah(teks):    # TODO 6.5 — kenali tiga kemungkinan keluaran model:    #   ("final", jawaban) | ("action", nama_tool, masukan) | ("gagal", teks)    # Perhatikan model sering membungkus nama tool dengan backtick atau kutip.    raise NotImplementedError# --- Loop --------------------------------------------------------------------def jalankan_agent(pertanyaan, maks_putaran=5, verbose=True):    # TODO 6.6 — loop utama agent:    #   1. panggil LLM dengan prompt berjalan    #   2. potong keluaran bila model terlanjur mengarang Observation sendiri    #   3. urai langkahnya    #   4. eksekusi tool, tempel hasilnya sebagai Observation, ulangi    #   5. berhenti saat Final Answer muncul ATAU batas putaran tercapai    # Batas putaran WAJIB ada, kalau tidak agent yang bingung akan berputar terus.    #    # Tangani juga tiga kasus kesalahan:    #   - model menyebut tool yang tidak ada    #   - parameternya salah bentuk    #   - keluarannya tidak bisa diurai sama sekali    #    # Kembalikan dict berisi jawaban, jejak, jumlah putaran, konteks, dan status.    raise NotImplementedError

In [ ]:
# TODO 6.7 (TUGAS LANGKAH 6) — pilih satu pertanyaan bertipe "multi_hop",# jalankan agent dengan verbose=True, dan tampilkan jejak lengkapnya.# Tunjukkan apakah agent benar-benar memanggil cari_dokumen LEBIH DARI SEKALI# dengan query yang berbeda. Simpan jejaknya ke hasil/l6_jejak_multihop.json

**Jawaban TUGAS LANGKAH 6** — jejak multi-hop dan analisisnya:_(tulis di sini)_

---## Langkah 7 — Agent VerifikatorAgent kedua memeriksa apakah tiap klaim di jawaban benar-benar didukung potongan yangdipakai. Bila ada yang tidak didukung, jawaban dikembalikan untuk diperbaiki, maksimalsatu kali putaran.Bandingkan biaya dan manfaatnya: multi-agent tidak selalu menang.

In [ ]:
# TODO 7.1 — system prompt untuk agent verifikator.# Tekankan: yang dinilai adalah DUKUNGAN DOKUMEN, bukan benar-salah menurut# pengetahuan model sendiri. Minta keluaran JSON dengan bentuk:#   {"didukung": bool, "klaim_tak_didukung": [...], "saran": "..."}SISTEM_VERIFIKATOR = ""def urai_json(teks):    # TODO 7.2 — model sering membungkus JSON dengan blok kode atau menambah    # kalimat pembuka. Bersihkan dulu, baru parse. Kembalikan None bila gagal.    raise NotImplementedErrordef verifikasi(pertanyaan, jawaban, potongan):    # TODO 7.3 — panggil verifikator dan kembalikan hasil yang sudah diurai.    # Bila keluaran gagal diurai, JANGAN diam-diam menganggapnya lolos.    raise NotImplementedErrordef pipeline_lengkap(pertanyaan, maks_perbaikan=1, verbose=False):    # TODO 7.4 — rangkai: agent menjawab, verifikator memeriksa, dan bila ada    # klaim tak didukung, jawaban dikembalikan untuk diperbaiki.    # Batasi perbaikan maksimal satu putaran.    raise NotImplementedError

In [ ]:
# TODO 7.5 (TUGAS LANGKAH 7) — bandingkan tiga konfigurasi pada dataset uji# yang sama: RAG sederhana (Langkah 5), agent tunggal (Langkah 6), dan# agent + verifikator (Langkah 7).## Untuk tiap konfigurasi catat: akurasi jawaban, total token, jumlah panggilan# LLM, dan latensi rata-rata. Manfaatkan dict TOTAL_TOKEN yang sudah dihitung# di dalam fungsi chat().## Kesimpulan yang harus ditulis: apakah kenaikan akurasi sebanding dengan# kenaikan biaya? Konfigurasi mana yang paling layak bila sistem ini benar-benar# dijalankan untuk layanan kampus?

**Jawaban TUGAS LANGKAH 7** — biaya vs manfaat multi-agent:_(tulis di sini)_

---## Langkah 8 — EvaluasiFaithfulness dihitung dalam tiga tahap: jawaban dipecah jadi klaim atomik, tiap klaimdinilai apakah bisa disimpulkan dari konteks, lalu skornya adalah rasio klaim terdukung.Bagian yang sering dilewatkan tapi penting: menguji keandalan jurinya sendiri, lewatpengulangan dan pengacakan urutan konteks.

In [ ]:
# TODO 8.1 — pecah jawaban menjadi klaim-klaim atomik memakai LLMdef pecah_klaim(jawaban):    # Sediakan cadangan: bila keluaran LLM tidak bisa diurai, pecah per kalimat.    raise NotImplementedError# TODO 8.2 — nilai satu klaim terhadap konteks.# Batasi keluaran ke tiga kemungkinan: ya, tidak, tidak_jelas.def nilai_klaim(klaim, potongan):    raise NotImplementedError# TODO 8.3 — skor faithfulness = rasio klaim terdukung terhadap total klaim.# Perhatikan kasus khusus: jawaban yang menolak menjawab tidak dinilai.def faithfulness(jawaban, potongan):    raise NotImplementedError# TODO 8.4 — hitung untuk subset dataset uji, tampilkan tabel, dan simpan.

In [ ]:
# TODO 8.5 (TUGAS LANGKAH 8) — uji keandalan jurinya sendiri. Tiga bagian:## (a) Konsistensi — jalankan penilaian yang sama tiga kali pada masukan yang#     sama. Apakah skornya berubah?## (b) Position bias — acak urutan potongan lalu nilai ulang. Apakah skornya#     ikut berubah? Kaitkan dengan temuan Zheng et al. (2023).## (c) Kesepakatan dengan manusia — nilai manual minimal 10 jawaban, lalu#     bandingkan dengan skor juri LLM dan hitung rata-rata selisihnya.## Kesimpulan yang harus ditulis: apakah skor juri pada sistem kalian layak# dipakai sebagai ANGKA MUTLAK, atau hanya layak sebagai PEMBANDING# antar-konfigurasi? Sertakan bukti angkanya.

**Jawaban TUGAS LANGKAH 8** — keandalan juri LLM:_(tulis di sini)_

---## Langkah 9 — Keamanan: Indirect Prompt InjectionSerangan di sini masuk lewat dokumen yang di-retrieve, bukan lewat input pengguna.Susupkan satu dokumen yang terlihat wajar tapi menyisipkan instruksi, buktikan agentmengikutinya, lalu pasang guardrail berlapis dan buktikan serangannya gagal.

In [ ]:
# TODO 9.1 — susun satu dokumen yang terlihat wajar tapi menyisipkan instruksi.# Contoh pola serangan ada di naskah praktikum bagian 9.1.DOKUMEN_JAHAT = ""# TODO 9.2 — salin korpus ke data/korpus_disusupi/, tambahkan dokumen jahat,# lalu bangun indeks kedua yang terpisah dari indeks bersih.# TODO 9.3 — ajukan pertanyaan yang membuat dokumen itu terambil.# Catat: apakah model mengikuti instruksi jahat tersebut?PERTANYAAN_SERANGAN = ""

In [ ]:
# TODO 9.4 — Lapis 1: pemisahan peran yang tegas.# Perkuat system prompt supaya isi di antara penanda <dokumen> diperlakukan# sebagai DATA yang dibaca, bukan PERINTAH yang dijalankan.SISTEM_AMAN = ""# TODO 9.5 — Lapis 2: penyaringan masukan.# Susun daftar pola instruksi mencurigakan, lalu saring potongan hasil retrieval# sebelum dimasukkan ke prompt.POLA_MENCURIGAKAN = []def saring_masukan(potongan):    raise NotImplementedError# TODO 9.6 — Lapis 3: penyaringan keluaran.# Tolak jawaban yang memuat nomor telepon atau tautan yang TIDAK ada di# dokumen sumber.def saring_keluaran(jawaban, potongan):    raise NotImplementedErrordef jawab_aman(pertanyaan, k=5):    # TODO 9.7 — rangkai ketiga lapis di atas    raise NotImplementedError# TODO 9.8 — uji ulang serangan yang sama. Apakah sekarang tertahan?

In [ ]:
# TODO 9.9 (TUGAS LANGKAH 9) — variasikan serangannya, minimal tiga variasi:#   - instruksi dalam bahasa Inggris#   - instruksi disisipkan di tengah paragraf panjang yang terlihat normal#   - instruksi disamarkan dengan tanda baca atau spasi di antara huruf## Untuk tiap variasi catat: tertahan di lapis mana, dan apakah serangannya# tetap berhasil.## Yang harus dijelaskan di laporan: mengapa pertahanan berbasis penyaringan# pola TIDAK AKAN PERNAH lengkap, dan bagaimana prinsip pembatasan akses tool# di Sub Bab XVI menjadi pertahanan yang lebih mendasar.

**Jawaban TUGAS LANGKAH 9** — serangan yang masih lolos dan mengapa:_(tulis di sini)_

---## PenutupSalin seluruh isi folder `hasil/` dan jawaban tugas tiap langkah ke laporan.Ingat bahwa bobot penilaian terbesar ada pada analisis kegagalan, bukan padasistem yang berjalan mulus.